In [5]:
"""
Database Incremental Load Script

- Connects to PostgreSQL source database
- Performs initial load to local SQLite
- Performs incremental load based on latest datetime
"""


import pandas as pd
from sqlalchemy import create_engine
import sqlite3
import yaml
from pathlib import Path

# Load Credentials from YAML
credentials_file = Path("credentials.yml")

with open(credentials_file, "r") as f:
    credentials = yaml.safe_load(f)

# PostgreSQL credentials   
user = credentials["postgresql"]["username"]
password = credentials["postgresql"]["password"]
host = 'cm-de-k1-db.c9ms60q6cw37.ap-southeast-2.rds.amazonaws.com'
port = '5432'
database = 'postgres'

# Database Connection (PostgreSQL)
connection_string = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'
engine = create_engine(connection_string)
connection = engine.connect()

Read data 

In [ ]:
schema = 'coffee'
table = 'sales'

query = f'SELECT * FROM {schema}.{table}'
df = pd.read_sql(query, connection)

df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


Load to storage (SQLite)

In [10]:
# Save data to local SQLite (initial load)
dest_database_file = "destination.db"
dest_conn = sqlite3.connect(dest_database_file)

# Save the full table
df.to_sql('coffee_sales', con=dest_conn, if_exists='replace', index=False)

print("Initial load completed. Table 'coffee_sales' created in SQLite.")

Initial load completed. Table 'coffee_sales' created in SQLite.


Incremental Loading

In [13]:
# Get latest datetime in storage (for incremental load)
latest_query = "SELECT MAX(datetime) as latest_date FROM coffee_sales"
des_latest_datetime = pd.read_sql(latest_query, dest_conn).iloc[0, 0]

# Filter incremental records from PostgreSQL
incremental_query = f"SELECT * FROM {schema}.{table} WHERE datetime > '{des_latest_datetime}'"
print(f"Lastest data is: {des_latest_datetime}")

# Read incremental data into DataFrame
df_incremental = pd.read_sql(incremental_query, connection)
print(f"Found {len(df_incremental)} new rows to load.")

# Append incremental rows to existing SQLite table
if not df_incremental.empty:
    df_incremental.to_sql('coffee_sales', dest_conn, if_exists='append', index=False)
    print(f"{len(df_incremental)} new rows appended to 'coffee_sales'.")
else:
    print("No new records to load.")

Lastest data is: 2025-03-23 18:11:38.635000
Found 0 new rows to load.
No new records to load.
